# Klasik Görüntü İşleme ile Mikroskobik Görüntülerde Hücre Segmentasyonu, Sayımı ve Zamansal Analizi

**Görüntü İşleme — Yüksek Lisans Ara Sınav Projesi**
**Öğrenci:** Murat Sağlam — **No:** 264327008
**Seed:** `27008 + 2026 = 29034` (öğrenci numarasının son 5 hanesi + 2026)
**Veri seti:** Cell Tracking Challenge — Fluo-N2DH-GOWT1 (eğitim arşivi)

Bu notebook ödevdeki adımları sırasıyla uygular. Bütün işlem kodları `src/cellcount/` modüllerindedir;
notebook bu modülleri çağırır, tabloları ve şekilleri üretir.

> **Test güvenliği:** Test (Sequence 02) referans sayıları yalnızca **Bölüm 12**'deki kilitleme adımından
> sonra okunabilir. `references.load_test_references()` fonksiyonu kilit dosyalarının SHA-256 özetlerini
> doğrulamadan çalışmaz.

## 0. Kurulum

In [ ]:
import sys, os, json, platform, urllib.request, zipfile
from pathlib import Path
REPO = Path.cwd()
sys.path.insert(0, str(REPO / "src"))
import pandas as pd, numpy as np, cv2, skimage, scipy, matplotlib
from IPython.display import Image, display
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 60)
from cellcount import config, data, selection, references, eda, experiments, validation, testing, figures, pipeline, posthoc
from cellcount import run_all as R
print("Python", platform.python_version(), "| OpenCV", cv2.__version__, "| scikit-image", skimage.__version__,
      "| SciPy", scipy.__version__, "| NumPy", np.__version__, "| pandas", pd.__version__)
print("Seed:", config.SEED, "| Veri klasörü:", config.DATA_DIR)

In [ ]:
# (Google Colab için) Veri seti yoksa resmi bağlantıdan indirilip data/ altına açılır.
DOWNLOAD_IF_MISSING = True
if not (config.DATA_DIR / "01").is_dir() and DOWNLOAD_IF_MISSING:
    config.DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    zpath = config.DATA_DIR.parent / "Fluo-N2DH-GOWT1.zip"
    print("İndiriliyor:", config.DATASET_URL)
    urllib.request.urlretrieve(config.DATASET_URL, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(config.DATA_DIR.parent)
data.check_dataset(); print("Veri seti hazır.")

## 2. Veri setinin ön incelemesi
Sequence 01 ve 02'nin boyut, veri tipi, yoğunluk ve histogram özellikleri. Hücre boyut/parlaklık
istatistikleri yalnızca Sequence 01 (geliştirme + doğrulama) anotasyonlarından hesaplanır.

In [ ]:
stats = R.stage_dataset()
stats

In [ ]:
display(Image(str(config.FIG_DIR / "fig_dataset_histograms.png"), width=700))

## 3. Geliştirme, doğrulama ve test görüntülerinin seçilmesi (seed = 29034)

In [ ]:
sel = R.stage_selection()
print("Seçim kuralları doğrulandı (çakışma yok, gruplar/bloklar arası >= 5 kare boşluk).")
sel

## 4. Referans hücre sayıları (geliştirme + doğrulama)

In [ ]:
refs, cells = R.stage_references(sel)
display(refs)
print(cells[["marker_area_px", "eq_diameter_px", "mean_intensity"]].describe().round(1))
print("Kenara değen referans hücre sayısı:", int(cells.touches_border.sum()))
print("Ortalama yoğunluğu < 10 olan (çok soluk) hücre oranı:", round(float((cells.mean_intensity < 10).mean()), 3))

## 5–10. Baseline ve kontrollü deneyler (yalnızca 10 geliştirme görüntüsü)
* **Bölüm 5 — Baseline:** gri seviye → Otsu → bağlı bileşen analizi → sayım. Minimum alan eşiği geliştirme MAE'sine göre seçilir.
* **Bölüm 6 — Yoğunluk dönüşümleri**, **Bölüm 7 — Filtreler**, **Bölüm 8 — Segmentasyon**, **Bölüm 9 — Morfoloji / ayırma**:
  her grupta sabit bir referans ayarı korunur ve yalnızca incelenen faktör değiştirilir.

In [ ]:
exp, exp_pf, baseline_min_area, calib = R.stage_experiments(sel)
print("Baseline için seçilen minimum alan:", baseline_min_area)
calib

In [ ]:
cols = ["experiment_id", "group", "changed_factor", "intensity", "filter", "segmentation", "morphology",
        "params", "count_MAE", "count_RMSE", "bias", "exact_acc", "mean_time_s", "missed", "false_pos",
        "merged_objs", "split_cells"]
exp[cols]

In [ ]:
for g, d in exp.groupby("group", sort=False):
    best = d.sort_values(["count_MAE", "count_RMSE"]).iloc[0]
    print(f"{g:16s} en iyi: {best.experiment_id} {best.changed_factor:40s} MAE={best.count_MAE:.2f}")

### Deney grafikleri

In [ ]:
config.ensure_dirs()
ctx = experiments.build_context(sel)
demo = sel[sel.group == "development"].iloc[3]
demo_ref = int(refs.set_index("image_id").loc[demo.image_id, "ref_count"])
figures.fig_experiments_mae(exp); figures.fig_param_sensitivity(exp, calib)
for p in ["fig_experiments_mae.png", "fig_parameter_sensitivity.png"]:
    display(Image(str(config.FIG_DIR / p), width=1000))

## 11. Doğrulama ve nihai pipeline seçimi

In [ ]:
for k, v in experiments.candidate_pipelines().items():
    print(f"{k}: {v['desc']}")
val_df, winner, final_doc = R.stage_validation(sel, baseline_min_area)
val_df

In [ ]:
print("Seçilen nihai pipeline:", winner)
print(json.dumps(final_doc["final_pipeline"], indent=2, ensure_ascii=False))
figures.fig_validation(val_df); figures.fig1_flow(final_doc)
display(Image(str(config.FIG_DIR / "fig_validation.png"), width=650))
display(Image(str(config.FIG_DIR / "fig1_final_pipeline_flow.png"), width=1000))

In [ ]:
# Görüntü içeren şekiller (rapor için; GitHub sürümünde çıktıları temizlenir)
paths = [figures.fig2_intensity(int(demo.frame), ctx), figures.fig_filters(int(demo.frame)),
         figures.fig3_seg_morph(int(demo.frame), demo_ref, ctx), figures.fig4_steps(final_doc, int(demo.frame), ctx),
         figures.fig_dataset_samples(int(demo.frame), int(sel[sel.group == "test"].frame.iloc[0]))]
for p in paths:
    display(Image(str(p), width=1000))

## 12. Nihai pipeline'ın kilitlenmesi ve test tahminlerinin kaydedilmesi
Bu hücre `final_pipeline.json` dosyasını okur ve 15 test görüntüsünde baseline ile nihai pipeline'ı
**test referanslarını okumadan** çalıştırır. Ardından `test_predictions_locked.csv` ve SHA-256 özetlerini
içeren `test_lock_manifest.json` dosyalarını yazar.

> **Bu noktada** `final_pipeline.json`, `test_predictions_locked.csv`, `test_lock_manifest.json` ve kaynak
> kodları **ayrı bir commit** ile GitHub'a gönderilir (bkz. `GIT_ADIMLARI.md`).

In [ ]:
locked = R.stage_lock(sel)
print(json.dumps(testing.verify_lock(), indent=2, ensure_ascii=False))
locked

## 13. Bağımsız test performansı (kilit sonrası)

In [ ]:
test_df, summary, temporal, obj_err = R.stage_evaluate(sel)
test_df[["frame_label", "image_id", "frame", "test_block", "ref_count", "baseline_pred", "final_pred",
         "baseline_abs_err", "final_abs_err"]]

In [ ]:
summary

In [ ]:
obj_err.groupby("method")[["missed", "false_pos", "merged", "merged_cells", "split"]].sum()

## 14. Ardışık görüntülerde zamansal analiz

In [ ]:
figures.fig5_temporal(test_df)
display(Image(str(config.FIG_DIR / "fig5_temporal_counts.png"), width=1000))
temporal

## 15. Başarılı / hatalı / zorlayıcı örnekler

In [ ]:
p = figures.fig6_examples(final_doc, test_df, ctx)
display(Image(str(p), width=1000))

### Kilit sonrası hata analizi (yalnızca yorum amaçlı)
Kilitli tahminler **değiştirilmez**. Aşağıdaki analiz, nihai pipeline'ın testte neden fazla saydığını
açıklamak için Sequence 01 ve 02'nin log-uzayı arka plan gürültüsünü ve yanlış pozitif nesnelerin şeklini
karşılaştırır.

In [ ]:
display(pd.read_csv(config.REPO_ROOT / "posthoc_background_noise.csv"))
display(pd.read_csv(config.REPO_ROOT / "posthoc_object_shapes.csv"))
# Seçilmeyen adayların testte durumu (yalnızca bilgi; seçim/kilit değişmez)
display(pd.read_csv(config.REPO_ROOT / "posthoc_candidates_on_test.csv"))

In [ ]:
# Doğrulama ve test karşılaştırması
v = val_df.set_index("pipeline")
s = summary.set_index("method")
pd.DataFrame({"Doğrulama MAE": [v.loc["Baseline", "val_MAE"], v.loc[winner, "val_MAE"]],
              "Test MAE": [s.loc["Baseline", "MAE"], s.loc["Nihai pipeline", "MAE"]],
              "Doğrulama RMSE": [v.loc["Baseline", "val_RMSE"], v.loc[winner, "val_RMSE"]],
              "Test RMSE": [s.loc["Baseline", "RMSE"], s.loc["Nihai pipeline", "RMSE"]]},
             index=["Baseline", f"Nihai ({winner})"])

Bütün sayısal sonuçlar repo kökündeki CSV dosyalarına yazılmıştır: `selected_frames.csv`, `experiments.csv`,
`validation_results.csv`, `final_pipeline.json`, `test_predictions_locked.csv`, `test_results.csv`,
`test_summary.csv`, `temporal_analysis.csv`. Yorum ve tartışma teknik raporda yer almaktadır.